In [ ]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
load_dotenv()

train = pd.read_parquet("../../../data/train_data.parquet")
test = pd.read_parquet("../../../data/test_data.parquet")

train_val = train[train.is_val == True]
print("Number of samples to process:", len(train_val), len(test))

train.head()

In [ ]:
import os
import logging
from rouge import Rouge
from openai import OpenAI


client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5-nano-2025-08-07"

class GECScore:
    def __init__(self, llm_model):
        """
        Args:
            llm_model (str): Name of the LLM model to use.
        """
        self.llm_model = llm_model
        self.rouge = Rouge()

    def _run_gec(self, text):
        """
        Runs the grammar error correction (GEC) model.
        """
        prompt = f"""
        You are a highly skilled grammar correction AI in multiple languages.
        You are provided with a text separated by <text></text> tags.
        Correct any grammatical, spelling, or punctuation errors in the text, doing only minimal changes necessary.
        Return ONLY the corrected text without any additional commentary.
        # Text to correct:
        <text>{text}</text>
        """
        return self.chat_with_openai(prompt, self.llm_model)
    
    def chat_with_openai(self, prompt, model=MODEL):
        try:
            completion = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "user", "content": prompt},
                ],
                reasoning_effort="minimal",
                seed=42
            )
            return completion.choices[0].message.content
        except Exception as e:
            logging.error(f"Error during LLM interaction: {e}")
            return None      

    def get_gecscore(self, text):
        """
        Processes a single item: runs GEC if needed, and computes ROUGE.
        """
        # 1. Generate gec_text if missing
        try:
            gec_text = self._run_gec(text)
        except Exception as e:
            logging.error(f"Error running GEC: {e}")

        # 2. Compute ROUGE score
        try:
            if gec_text:
                scores = self.rouge.get_scores(text, gec_text, avg=True)
                gec_score = scores['rouge-2']['f']
        except Exception as e:
            logging.warning(f"ROUGE computation failed: {e}")
            gec_score = None
        return gec_score

# Processing GEC scores for the validation set: 

In [ ]:
import concurrent.futures
import json
import pandas as pd
from tqdm import tqdm

def get_gecscore_for_row(row):
    """
    Function to get GEC score for a single row.
    Reurns a dictionary with key (model, pipeline, user_id, thread_id) and (score_real, score_generated).
    """
    model = row['model']
    pipeline = row['pipeline']
    user_id = row['user_id']
    thread_id = row['thread_id']
    
    text_real = row['real_text']
    text_generated = row['generated_text']
    
    try:
        score_real = gec_scorer.get_gecscore(text_real)
    except Exception as e:
        logging.error(f"Error getting GEC score for real text: {e}")
        score_real = None   
    
    try:
        score_generated = gec_scorer.get_gecscore(text_generated)
    except Exception as e:
        logging.error(f"Error getting GEC score for generated text: {e}")
        score_generated = None
    
    return {
        (model, pipeline, user_id, thread_id): (score_real, score_generated)
    }

# test with a row: 
sample_row = train_val.iloc[1]
print(get_gecscore_for_row(sample_row))

In [ ]:
all_scores = {}
with concurrent.futures.ThreadPoolExecutor(max_workers=64) as executor:
    # Submit each row (as a Series) for processing
    futures = {executor.submit(get_gecscore_for_row, row): index for index, row in train_val.iterrows()}
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Gec Score Computation"):
        result = future.result()
        if result is not None:
            all_scores.update(result)

# Save results
import joblib
joblib.dump(all_scores, "gec_rouge2_scores_train_val.pkl")

In [ ]:
# Save results
import joblib

all_scores_test = {}
with concurrent.futures.ThreadPoolExecutor(max_workers=64) as executor:
    # Submit each row (as a Series) for processing
    futures = {executor.submit(get_gecscore_for_row, row): index for index, row in test.iterrows()}
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Gec Score Computation"):
        result = future.result()
        if result is not None:
            all_scores_test.update(result)
        
        if len(all_scores_test) % 1000 == 0:
            print(f"Processed {len(all_scores_test)} / {len(test)} rows.")
            joblib.dump(all_scores_test, "gec_rouge2_scores_test.pkl")

joblib.dump(all_scores_test, "gec_rouge2_scores_test.pkl")